# Module 5 — Integration, stress testing and defense


This is the complete teaching lab for Module 5. Run setup first, then follow the cells in order. Section headings are learning topics; [the current lesson map](../../docs/LESSON_MAP.md) maps them to the course videos. Collapsing a section does not skip execution.

**Practice versus assessment.** Worked examples teach the method; they are not completed capstone evidence. Use the [RiskLens capstone repository](https://github.com/SarasAI-Institute/Building-Custom-AI-Capstone) for exact evidence thresholds and all 24 Met / Not Yet criteria. Synthetic examples, tiny practice splits and unreviewed judge outputs do not establish mastery.

**Runtime.** Read [Colab setup and artifact handoffs](../../docs/COLAB.md). The CPU exercises run separately in `practice.ipynb` and `solutions.ipynb`. This lab uses model downloads and GPU execution. Modules 3–5 require your saved artifacts and explicit train/load choices. A notebook that stops for missing artifacts is not a completed run. See [what was tested](../../docs/VALIDATION.md).


## Shared setup before running

In [ ]:
"""Run this cell first in a fresh Colab session, before importing model libraries."""
import importlib.metadata as _metadata
import platform as _platform
import subprocess as _subprocess
import sys as _sys
import tempfile as _tempfile
from pathlib import Path as _Path

COURSE_PACKAGES = [
    "transformers==5.22.0", "trl==1.17.0", "peft==0.21.2",
    "accelerate==1.15.0", "datasets==5.1.0", "sentence-transformers==6.2.0",
    "faiss-cpu==1.15.1", "num2words==0.5.14",
    "pillow==13.0.0", "scikit-learn==1.9.1", "matplotlib==3.11.2", "requests==2.34.2",
]
if _sys.version_info < (3, 11):
    raise RuntimeError("Use a current Colab Python 3.11+ runtime.")
try:
    _torch_version = _metadata.version("torch")
except _metadata.PackageNotFoundError as _error:
    raise RuntimeError("Use Colab's preinstalled matched PyTorch/torchvision runtime; see docs/COLAB.md.") from _error
_constraints = ["torch==" + _torch_version]
try:
    _constraints.append("torchvision==" + _metadata.version("torchvision"))
except _metadata.PackageNotFoundError:
    raise RuntimeError("The vision labs need a matching torchvision installation; use a fresh Colab runtime.")
if _platform.system() == "Linux":
    COURSE_PACKAGES.append("bitsandbytes==0.50.2")
_before = {}
for _name in ["transformers", "trl", "peft", "datasets", "sentence_transformers", "numpy"]:
    if _name in _sys.modules:
        _before[_name] = getattr(_sys.modules[_name], "__version__", None)
with _tempfile.TemporaryDirectory() as _temporary:
    _constraint_file = _Path(_temporary) / "preserve-colab-torch.txt"
    _constraint_file.write_text("\n".join(_constraints))
    _subprocess.run([_sys.executable, "-m", "pip", "install", "--quiet", "--upgrade-strategy", "only-if-needed",
                     "-c", str(_constraint_file), *COURSE_PACKAGES], check=True)
if _before:
    raise RuntimeError("Packages installed. Restart the Colab session once, then run from the setup cell again before importing models.")
print("Course stack installed; existing PyTorch/torchvision CUDA wheels preserved.")
print("If installation failed, stop here. Start a fresh GPU runtime and inspect the resolver message.")


In [ ]:
import torch, json, time, os
from pathlib import Path
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
assert torch.cuda.is_available(), "Use a GPU runtime for this combined model demonstration."


### Restore the actual selected components
Before running, create `risklens_config.json` from the template below. Copy the exact reviewed inspector prompt, processor resize settings and revision from Module 1, frozen corpus from Module 2, and evaluated folded Analyst artifact from Module 4. This bridge supports the folded-half, folded-INT8 and folded-NF4 paths; another selected architecture needs its own validated loader. A populated configuration is not evidence of passing the project.

Required configuration structure (replace nulls with real recorded values):
```json
{"inspector":{"model_id":"HuggingFaceTB/SmolVLM2-2.2B-Instruct","revision":null,"prompt":null,"processor_kwargs":{},"max_new_tokens":120},"corpus_path":"rag_practice_snapshot_v1.json","source_products_path":"source_products.json","chunk_words":120,"chunk_overlap":30,"retrieval_k":4,"embedding_revision":null,"analyst":{"path":"analyst-merged","arm":null,"compute_dtype":null,"review_record":null},"context_budget":2048}
```
`source_products.json` maps each source ID to a human-verified list of applicable product IDs. Do not assign generic reviews to an appliance without provenance. For practice, the source IDs `spec_kettle`, `spec_toaster`, `spec_blender` and `policy_returns` can be mapped to their explicit synthetic catalogue identities after checking their actual text. No bottle-to-kettle substitution is made. The project uses the official corpus, not a silently substituted workshop snapshot.

The inspector contract requires JSON with verdict GOOD or DEFECTIVE, defect_type string or null, and a nonempty description. If the selected classifier produces only a binary verdict, evaluate and freeze the structured-note prompt separately before integration. Preserve that change in the record.


## Topic — Overview of Module 5: What a Complete RiskLens Must Demonstrate

**Theory · 6–8 minutes**

Frames the final delivery and distinguishes component quality from system behavior. Explains the required real-input demonstrations, evidence trace and final portfolio so learners know what completion looks like.



### Evidence at every interface

Preserve image identity and hash, externally supplied product identity, raw and parsed inspection, predicted route, query, returned full source texts and final report. Human reference labels remain separate from model predictions. A GOOD short-circuit is a model decision; a PARSE_ERROR is a recorded failure, never a good product.

**Illustrative arithmetic:** among 100 true-defect inputs, 90 satisfy correct inspection/routing, 72 also have adequate matching evidence, and 54 also have acceptable reports. The all-stages-success event has rate (90/100) × (72/90) × (54/72) = .54. These are conditional rates from the same cohort, not independent benchmark scores. If your task permits recovery, define and measure its final success separately.

A separate ten-case illustration has six supported outcomes, two unsupported reports, one parse failure and one missed defect. Keep all ten in the declared success denominator: 6/10 = .60. These are not actual course-model measurements.

One illustrative warm sequential request takes .6 s inspection + .1 s retrieval + 1.2 s Analyst + .1 s wrapper = 2.0 s. Queueing and model load are outside that boundary. Measure whole requests directly; do not add separate stage percentiles.

**Project evidence:** D4.5 requires at least three real product demonstrations, including one good and two defective products. D4.7 requires at least ten products, end-to-end p50/p95, stage times and a failure taxonomy. D4.6 and final delivery add a justified runtime, scale limitations and monitoring, accessible artifacts, a clean run/load, metrics ledger and reflection of at most 300 words. C5.1–C5.4 map these records to integration, evaluation, reproducibility and explanation; final exam procedures remain pending.


## Topic — Assemble the Real RiskLens Pipeline

**Tutorial: Colab · 6–8 minutes**

Replaces the lesson stubs with the learner’s inspector and retriever, connects the chosen Analyst and traces one image through evidence retrieval to a report. Shows where to inspect intermediate outputs when the report is wrong.



### Load the inspector

Uses the same model and prompt contract as the workshop. For assessed work replace the model/prompt configuration with your frozen Module 1 selection.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import hashlib

def stable_sha(x):
    return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False).encode()).hexdigest()

def file_sha(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda:stream.read(1024*1024),b""): h.update(block)
    return h.hexdigest()

config_path=Path("risklens_config.json")
if not config_path.exists(): raise FileNotFoundError("Prepare the recorded component configuration above before running.")
CONFIG=json.loads(config_path.read_text()); ic=CONFIG["inspector"]
if not all(isinstance(ic.get(k),str) and ic[k].strip() for k in ["model_id","revision","prompt"]):
    raise ValueError("Exact inspector revision and reviewed prompt required")
if type(ic["max_new_tokens"]) is not int or ic["max_new_tokens"]<1: raise ValueError("Positive generation limit required")
processor=AutoProcessor.from_pretrained(ic["model_id"],revision=ic["revision"],**ic["processor_kwargs"])
vlm=AutoModelForImageTextToText.from_pretrained(ic["model_id"],revision=ic["revision"],dtype=DTYPE,device_map={"":"cuda:0"})
vlm.eval()
print("Inspector configuration",stable_sha(ic))


In [ ]:
def ask_vlm(image, question, max_new_tokens=120):
    messages=[{"role":"user","content":[{"type":"image","image":image},{"type":"text","text":question}]}]
    inputs=processor.apply_chat_template(messages,add_generation_prompt=True,tokenize=True,return_dict=True,return_tensors="pt")
    inputs={k:v.to(vlm.device,dtype=DTYPE) if torch.is_floating_point(v) else v.to(vlm.device) for k,v in inputs.items()}
    prompt_length=int(inputs["input_ids"].shape[1])
    with torch.inference_mode():
        out=vlm.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False)
    new_ids=out[0,prompt_length:].detach().cpu().tolist()
    return {"raw":processor.decode(new_ids,skip_special_tokens=True).strip(),"new_ids":new_ids,
        "reached_token_limit":len(new_ids)>=max_new_tokens}


In [ ]:
def parse_inspection(raw):
    try:
        note=json.loads(raw)
        if not isinstance(note,dict) or set(note)!={"verdict","defect_type","description"}: raise ValueError("Unexpected schema")
        if note["verdict"] not in {"GOOD","DEFECTIVE"}: raise ValueError("Invalid verdict")
        if note["defect_type"] is not None and not isinstance(note["defect_type"],str): raise ValueError("Invalid defect type")
        if not isinstance(note["description"],str) or not note["description"].strip(): raise ValueError("Description required")
        if note["verdict"]=="DEFECTIVE" and not (isinstance(note["defect_type"],str) and note["defect_type"].strip()): raise ValueError("Defect label required")
        return {**note,"raw":raw,"parse_status":"valid"}
    except (ValueError,TypeError,KeyError) as exc:
        return {"verdict":"PARSE_ERROR","raw":raw,"parse_status":"invalid","error":str(exc)}

def inspect(image):
    result=ask_vlm(image,ic["prompt"],ic["max_new_tokens"])
    return {**parse_inspection(result["raw"]),**result}


### Restore the evidence corpus and index

This rebuilds the workshop evidence corpus. For the project load your frozen corpus and selected retrieval configuration instead. Source IDs must remain stable across evaluation and demonstration.

In [ ]:
snapshot_path=Path(CONFIG["corpus_path"])
snapshot=json.loads(snapshot_path.read_text())
corpus=snapshot["corpus"]
if not corpus or len({d["source"] for d in corpus})!=len(corpus): raise ValueError("Nonempty corpus with unique source IDs required")
if "sha256" in snapshot and stable_sha({"corpus":corpus,"gold":snapshot["gold"]})!=snapshot["sha256"]:
    raise ValueError("Frozen snapshot hash mismatch")
source_products=json.loads(Path(CONFIG["source_products_path"]).read_text())
known_sources={d["source"] for d in corpus}
if not set(source_products)<=known_sources: raise ValueError("Mapping contains unknown sources")
if any(not isinstance(ids,list) or any(not isinstance(i,str) or not i.strip() for i in ids) for ids in source_products.values()):
    raise ValueError("Product mapping must contain explicit lists of nonempty IDs")
print("Restored",len(corpus),"documents; corpus file SHA",file_sha(snapshot_path))


In [ ]:
# Keep the exact frozen texts. Unmapped sources remain unavailable for product-specific claims.
print("Mapped sources",len(source_products),"of",len(corpus))
for source,products in source_products.items(): print(source,products)


In [ ]:
def chunk(text, source, max_words=120, overlap=30):
    if type(max_words) is not int or type(overlap) is not int or not 0<=overlap<max_words:
        raise ValueError("Require 0 <= overlap < max_words")
    words=text.split(); out=[]; start=0
    while start<len(words):
        end=min(start+max_words,len(words))
        out.append({"source":source,"chunk_id":f"{source}:{start}:{end}","word_start":start,"word_end":end,"text":" ".join(words[start:end])})
        if end==len(words): break
        start=end-overlap
    return out
chunks=[{**c,"kind":d.get("kind","doc")} for d in corpus for c in chunk(d["text"],d["source"],CONFIG["chunk_words"],CONFIG["chunk_overlap"])]
if not chunks: raise ValueError("No chunks")


In [ ]:
from sentence_transformers import SentenceTransformer
if not CONFIG.get("embedding_revision"): raise ValueError("Record the selected embedding model revision")
embedder=SentenceTransformer("BAAI/bge-small-en-v1.5",revision=CONFIG["embedding_revision"],device="cpu")
vectors=embedder.encode([c["text"] for c in chunks],normalize_embeddings=True)
print("Embedding matrix",vectors.shape)


In [ ]:
import faiss
import numpy as np
vectors=np.asarray(vectors,dtype=np.float32)
if vectors.ndim!=2 or len(vectors)!=len(chunks) or not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors,axis=1),1,atol=1e-4):
    raise ValueError("Finite normalized vectors aligned with chunks required")
index=faiss.IndexFlatIP(vectors.shape[1]); index.add(vectors)
BGE_PREFIX="Represent this sentence for searching relevant passages: "

def retrieve(question,k=4,product_id=None):
    if type(k) is not int or k<1: raise ValueError("Positive integer k required")
    if not isinstance(product_id,str) or not product_id.strip(): raise ValueError("Externally supplied product ID required")
    eligible=[i for i,c in enumerate(chunks) if product_id in source_products.get(c["source"],[])]
    if not eligible: return []
    q=embedder.encode([BGE_PREFIX+question],normalize_embeddings=True).astype(np.float32)
    if not np.isfinite(q).all() or not np.allclose(np.linalg.norm(q,axis=1),1,atol=1e-4): raise ValueError("Invalid query vector")
    # Exact filtered cosine search; no unrelated product can enter through a high score.
    scores=vectors[eligible]@q[0]; order=np.argsort(-scores,kind="stable")[:min(k,len(eligible))]
    return [{**chunks[eligible[j]],"score":float(scores[j])} for j in order]


### Load the selected Analyst artifact

Set the local path to the actual merged checkpoint you selected in Module 4. Missing artifacts stop here rather than silently substituting the unadapted base. Loading this checkpoint at 4-bit is a candidate configuration that must match the configuration evaluated in your gate.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
ac=CONFIG["analyst"]; ANALYST_PATH=Path(ac["path"])
if ac["arm"] not in {"folded_half","folded_int8","folded_nf4"}: raise ValueError("Use the evaluated folded arm or implement and evaluate the selected architecture's loader")
if ac.get("compute_dtype")!=str(DTYPE): raise ValueError("Runtime compute dtype differs from selected evaluation")
review_path=Path(ac["review_record"])
review=json.loads(review_path.read_text())
if review.get("decision") not in {"ACCEPT","FALLBACK"} or not review.get("chosen_artifact"):
    raise ValueError("Complete Module 4 review: ACCEPT or a justified evaluated FALLBACK")
if review.get("chosen_arm")!=ac["arm"]: raise ValueError("Chosen quantization arm differs from configuration")
if any(not review.get(k) for k in ["reason","claim_support_review","severity_and_action_review","runtime_acceptance"]):
    raise ValueError("Human quality and runtime review must be complete")
comparison_plan=json.loads((review_path.parent/"plan.json").read_text())
gate_record=json.loads((review_path.parent/"gate_results.json").read_text())
if review["plan_sha256"]!=stable_sha(comparison_plan) or gate_record["plan_sha256"]!=stable_sha(comparison_plan):
    raise ValueError("Decision and gate must refer to the same frozen plan")
chosen_gate=gate_record["results"].get(ac["arm"],{})
if not chosen_gate.get("passes"): raise ValueError("This loader only accepts an evaluated passing folded arm, including a passing half-precision fallback")
actual_files={str(x.relative_to(ANALYST_PATH)):file_sha(x) for x in sorted(ANALYST_PATH.rglob("*")) if x.is_file() and x.name!="reload_check.json"}
if actual_files!=comparison_plan["folded_files"]: raise ValueError("Actual checkpoint differs from evaluated artifact")
if Path(review["chosen_artifact"]).resolve()!=ANALYST_PATH.resolve(): raise ValueError("Chosen artifact path differs")
if not (ANALYST_PATH/"fold_manifest.json").exists(): raise FileNotFoundError("Restore the folded checkpoint and provenance manifest")
kwargs={"device_map":{"":"cuda:0"},"torch_dtype":DTYPE}
if ac["arm"]=="folded_nf4":
    kwargs["quantization_config"]=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=False,bnb_4bit_compute_dtype=DTYPE)
if ac["arm"]=="folded_int8":
    kwargs["quantization_config"]=BitsAndBytesConfig(load_in_8bit=True)
tok=AutoTokenizer.from_pretrained(str(ANALYST_PATH),local_files_only=True)
analyst_4bit=AutoModelForCausalLM.from_pretrained(str(ANALYST_PATH),local_files_only=True,**kwargs)
analyst_4bit.eval()  # Legacy variable name retained; actual load mode is ac['arm'].
artifact_files={str(x.relative_to(ANALYST_PATH)):file_sha(x) for x in sorted(ANALYST_PATH.rglob("*")) if x.is_file()}
print("Loaded",ac["arm"],"artifact identity",stable_sha(artifact_files))


In [ ]:
import re
REQUIRED=["## ROOT CAUSE ANALYSIS","**Product:**","**Defect:**","**Severity:**","**Evidence:**","**Probable cause:**","**Recommended action:**"]
def format_ok(text):
    """Mechanical format only: all headers and one exact LOW/MEDIUM/HIGH severity line."""
    if not isinstance(text, str) or not all(h in text for h in REQUIRED):
        return False
    lines = re.findall(r"(?m)^\*\*Severity:\*\*[^\n]*$", text)
    return len(lines) == 1 and bool(re.fullmatch(
        r"\*\*Severity:\*\*[ \t]*(LOW|MEDIUM|HIGH)[ \t]*", lines[0]))

def chat(model,text,max_new_tokens=350):
    inputs=tok.apply_chat_template([{"role":"user","content":text}],add_generation_prompt=True,return_dict=True,return_tensors="pt").to(model.device)
    length=int(inputs["input_ids"].shape[1])
    if length+max_new_tokens>CONFIG["context_budget"]: raise ValueError("Prompt plus output reserve exceeds declared context budget")
    with torch.inference_mode():
        out=model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tok.eos_token_id)
    ids=out[0,length:].detach().cpu().tolist()
    return {"text":tok.decode(ids,skip_special_tokens=True).strip(),"new_ids":ids,"prompt_tokens":length,"reached_token_limit":len(ids)>=max_new_tokens}


In [ ]:
def make_input(defect_note,evidence):
    return (f"Defect note from visual inspection:\n{defect_note}\n\n"
            f"Retrieved evidence:\n{evidence}\n\nWrite a root cause analysis report.")

def sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()

def analyze_product_trace(image,product_id):
    if image is None or not isinstance(product_id,str) or not product_id.strip():
        raise ValueError("Real image and externally verified product ID required")
    record={"product_id":product_id,"status":"started","timing":{},"hits":[],"evidence":"","report":"","human_review":None}
    sync(); started=time.perf_counter(); stage="inspection"
    try:
        t=time.perf_counter(); note=inspect(image); sync(); record["timing"]["inspection_s"]=time.perf_counter()-t; record["note"]=note
        if note["verdict"]=="PARSE_ERROR": record["status"]="inspection_error"; return record
        if note["verdict"]=="GOOD": record["status"]="no_defect_predicted"; return record
        stage="retrieval"; t=time.perf_counter()
        record["query"]=f"{product_id} {note['defect_type']} {note['description']}"
        hits=retrieve(record["query"],CONFIG["retrieval_k"],product_id); record["hits"]=hits
        record["timing"]["retrieval_s"]=time.perf_counter()-t
        if not hits: record["status"]="no_matching_evidence"; return record
        evidence="\n".join(f"[{h['source']}] {h['text']}" for h in hits)
        record["evidence"]=evidence
        record["analyst_input"]=make_input(f"Product {product_id}. {note['defect_type']}: {note['description']}",evidence)
        stage="analyst"; t=time.perf_counter(); generated=chat(analyst_4bit,record["analyst_input"]); sync()
        record["timing"]["analyst_s"]=time.perf_counter()-t;record["generation"]=generated;record["report"]=generated["text"]
        record["format_ok"]=format_ok(generated["text"]);record["status"]="report_generated"
        return record
    except Exception as exc:
        record.update(status="runtime_error",failed_stage=stage,error=repr(exc)); return record
    finally:
        sync(); record["total_s"]=time.perf_counter()-started
        record["timing_scope"]="warm image-object request; preprocessing, retrieval, generation and wrapper; model load, image-file decode and queue excluded"

def analyze_product(image, product_id=None):
    """Capstone-compatible dict. Provide product identity through image.info or the optional argument."""
    product_id=product_id or getattr(image,"info",{}).get("product_id")
    trace=analyze_product_trace(image,product_id)
    return {"note":trace.get("note",{}),"evidence":trace.get("evidence",""),
            "report":trace.get("report",""),"trace":trace}


### Demonstrate three real products
Use `from PIL import Image`, open an actual image with a context manager, convert to RGB, then call `analyze_product(image, product_id="YOUR_VERIFIED_ID")`. Save the full returned dictionary alongside image path/hash and a separate human reference label. Demonstrate at least one good and two defective products for D4.5. Inspect raw output, parsed verdict, matching source texts, complete report and timing. GOOD is a prediction, not a human quality guarantee. A generated report remains unreviewed until its claims and recommendations are checked. GPU inference was not run during artifact production; rehearse the combined memory load before filming.


## Topic — Stress-Test RiskLens & Prepare a Reproducible Handoff

**Tutorial: Colab · 6–8 minutes**

Runs a prepared small batch, reads the stage-time and failure summary, then checks a clean run/load and metrics ledger. Shows the deployment-note essentials: runtime choice, scale limitations, monitoring and the assumptions behind the measurements.



### Freeze the system workload before running
Create `risklens_batch_manifest.json` as a list of records with unique `case_id`, `image_path`, `product_id` and human `reference_verdict` (GOOD or DEFECTIVE). Use at least ten distinct real product cases for D4.7. The label is used only for evaluation, never passed into the pipeline. Preserve image identity and acquisition provenance outside the runtime.

The batch below measures file decode plus the warm full pipeline, with model loading and serving queue excluded. The inner trace additionally measures the image-object boundary. Use two declared, excluded warmups; retain them separately. Missing images, parse errors, empty evidence and runtime failures remain visible. Stage percentiles describe only cases where that stage completed and report coverage; do not substitute zero for an unrun stage or add stage p95 values.

This sequential diagnostic batch is not a concurrent load test. Describe the workload actually measured. For a new trial, choose a new output folder; do not overwrite the earlier run.


In [ ]:
from PIL import Image
import numpy as np
import platform, importlib.metadata
BATCH_ROOT=Path("risklens_batch_v1");BATCH_ROOT.mkdir(exist_ok=True)
manifest=json.loads(Path("risklens_batch_manifest.json").read_text())
if not isinstance(manifest,list) or len(manifest)<10: raise ValueError("D4.7 requires at least ten real product cases")
if len({x["case_id"] for x in manifest})!=len(manifest): raise ValueError("Unique case IDs required")
if any(x["reference_verdict"] not in {"GOOD","DEFECTIVE"} or not x["product_id"] for x in manifest): raise ValueError("Valid human labels and product IDs required")
# Hash available inputs, retaining missing-file cases rather than silently excluding them.
cases=[{**x,"image_sha256":file_sha(x["image_path"]) if Path(x["image_path"]).is_file() else None} for x in manifest]
plan={"cases":cases,"config":CONFIG,"config_sha256":stable_sha(CONFIG),"artifact_files":artifact_files,
      "corpus_sha256":file_sha(CONFIG["corpus_path"]),"source_products_sha256":file_sha(CONFIG["source_products_path"]),
      "warmups":2,"warmup_case_id":cases[0]["case_id"],"batch_size":1,
      "timing_scope":"file decode plus warm pipeline; model load and serving queue excluded; synchronized CUDA",
      "device":torch.cuda.get_device_name(),"python":platform.python_version(),
      "packages":{k:importlib.metadata.version(k) for k in ["torch","transformers","peft","bitsandbytes","sentence-transformers","numpy"]}}
plan_path=BATCH_ROOT/"plan.json"
if plan_path.exists():
    if json.loads(plan_path.read_text())!=plan: raise ValueError("Plan changed; preserve this run and choose a new folder")
else: plan_path.write_text(json.dumps(plan,indent=2))

def run_case(case):
    sync(); start=time.perf_counter()
    try:
        with Image.open(case["image_path"]) as opened: image=opened.convert("RGB")
        result=analyze_product_trace(image,case["product_id"])
    except Exception as exc:
        result={"status":"input_or_runtime_error","error":repr(exc),"timing":{}}
    sync(); elapsed=time.perf_counter()-start
    return {**case,**result,"elapsed_s":elapsed,"review":{"case_success":None,"primary_failure":None,"claim_support":None,"notes":None}}

result_path=BATCH_ROOT/"results.json"
if result_path.exists():
    saved=json.loads(result_path.read_text())
    if saved["plan_sha256"]!=stable_sha(plan): raise ValueError("Result plan mismatch")
    if not saved["complete"]: raise RuntimeError("Partial run retained; inspect it before choosing a new run folder")
    records=saved["records"]; print("Restored complete saved batch without new inference")
else:
    warmups=[run_case(cases[0]) for _ in range(plan["warmups"])]
    (BATCH_ROOT/"warmups.json").write_text(json.dumps(warmups,indent=2))
    records=[]
    for case in cases:
        records.append(run_case(case))
        result_path.write_text(json.dumps({"plan_sha256":stable_sha(plan),"complete":len(records)==len(cases),"records":records},indent=2))

def percentile_summary(values):
    if not values: return {"n":0,"mean_s":None,"p50_s":None,"p95_s":None}
    a=np.asarray(values,dtype=float)
    if not np.isfinite(a).all() or (a<0).any(): raise ValueError("Finite nonnegative durations required")
    return {"n":len(a),"mean_s":float(a.mean()),"p50_s":float(np.percentile(a,50)),"p95_s":float(np.percentile(a,95)),"method":"linear"}

summary={"all_attempts":percentile_summary([x["elapsed_s"] for x in records]),
    "status_counts":{s:sum(x["status"]==s for x in records) for s in sorted({x["status"] for x in records})},
    "stages":{s:percentile_summary([x["timing"][s] for x in records if s in x.get("timing",{})]) for s in ["inspection_s","retrieval_s","analyst_s"]},
    "limits":"Stage coverage differs by routing/failure. No population p95 guarantee or concurrency claim.",
    "case_success_rate":None,"human_review":"PENDING"}
(BATCH_ROOT/"summary.json").write_text(json.dumps(summary,indent=2))
review_path=BATCH_ROOT/"human_review.json"
if not review_path.exists():
    review_path.write_text(json.dumps({"plan_sha256":stable_sha(plan),"success_rule":None,"rows":[{"case_id":x["case_id"],**x["review"]} for x in records]},indent=2))
print(json.dumps(summary,indent=2))
for x in records: print(x["case_id"],x["reference_verdict"],x.get("note",{}).get("verdict"),x["status"],x.get("report"))


### Review failures and prepare the reproducible handoff
Open every full result and fill `human_review.json`. Define the success rule before classifying cases. A true GOOD case needs a correct short-circuit; a true defective case needs appropriate inspection, applicable evidence and an acceptable supported report under your declared rule. Treat parsing and runtime failures as unsuccessful attempts. Keep failed cases in the denominator, retain overlapping diagnoses in notes and use a primary failure category for a readable summary. A generated report or a format pass does not automatically count as success.

For a reviewed rate, require exactly one Boolean `case_success` per planned case; sum True values and divide by all planned cases. Keep the result unavailable while any review is missing. Report inspection confusion counts separately and retain invalid predictions. Never use the reference label as a model input.

**Delivery checklist:**
- Save `risklens_config.json`, source-to-product mapping, exact frozen corpus, all model/tokenizer artifacts and their identities, package versions and hardware record.
- Save batch manifest, plan, excluded warmups, complete raw results, all-attempt and per-stage summaries, human review and failure analysis.
- Link the Module 1 evaluation, Module 2 retrieval/judge/refusal evidence, Module 3 paired Analyst evaluation and ablation, and Module 4 compression comparison into one metrics ledger. Every value needs its source file, sample size, split, metric definition and runtime boundary.
- Write the D4.6 deployment note: intended runtime, likely bottleneck at 100× requests, what would need testing, monitoring signals and known quality limits. A sequential ten-case batch does not prove scale.
- In a fresh runtime, install recorded dependencies, restore artifacts, execute shared setup then Videos 02–03, and preserve a new replay record. Compare under declared tolerances; do not mark replay passed until performed. Confirm another reviewer can access all linked artifacts.
- Write a reflection of at most 300 words grounded in actual tradeoffs, failed experiments and remaining limitations. Use the next video's criterion map to prepare your explanation.

No clean GPU replay, load test or model inference was executed during this material's production. The code is prepared for the instructor's real artifacts and recording environment.


## Topic — Explaining the Final Project & Preparing Your Defense

**Theory/Demo · 6–8 minutes**

Walks through the final submission and a short example of explaining a component, tracing a product and defending one measurement. Shows how to acknowledge a failed experiment or unsupported conclusion. Exam-specific instructions will be finalized with the exam brief.



### Published competency evidence map

The supplied v2.0 rubric governs these criteria. The tutorial illustrates methods; the learner must supply their own actual project evidence. Review the must-pass criterion first.

| Criterion | Evidence | Met standard |
|---|---|---|
| C5.1 **Must-pass** — Your real components work together | D4.5 analyze_product(); three saved product demonstrations; selected Analyst record. | Compose the learner inspector, retriever and selected Analyst through the published contracts. Preserve the GOOD short-circuit and return note, evidence and report. Save at least three product demonstrations: one good and two defective, with traceable inputs and outputs. Use real components, not hard-coded substitutes. Label any fallback or failed-gate demonstration and its limits. |
| C5.2 — Your system evaluation locates failures | D4.7 batch outputs; timing table; product-level failure taxonomy. | Run the full pipeline on at least ten products and save end-to-end p50/p95 plus inspector, retrieval and Analyst timing. For every wrong or weak report, identify the responsible stage and evidence for that diagnosis; explicitly report if none are observed. Explain what the small workload cannot establish about production reliability or generalization. |
| C5.3 — Your delivery is reproducible and documented | D4.6 deployment note; final run/load and show_ledger(); artifacts, environment and reflection. | Provide a clean top-to-bottom run/load with saved evaluation outputs, accessible model/data artifacts and environment/configuration details. Display the complete metrics ledger and preserve earlier comparisons separately. Document runtime choice, the bottleneck at 100 times traffic and a monitoring signal. Include a reflection of at most 300 words on trade-offs, trusted/uncertain numbers and next steps. |
| C5.4 — You can explain the submitted system | Explanation cells in final notebook; traceable code/outputs; exam evidence if separately issued. | Explain what one actual component does, why its design fits RiskLens and what would break if a named part were removed. Trace one demonstrated product through the applicable stages. Defend one reported metric and discuss a failed experiment or unsupported claim using saved evidence. Submit a written walkthrough; separately issued exam evidence may replace equivalent parts. |

All criteria must be Met for this competency to be Mastered. See [practice-to-capstone differences](../../docs/ASSESSMENT.md) and the [exact consolidated rubric](https://github.com/SarasAI-Institute/Building-Custom-AI-Capstone/blob/codex/competency-colab-refresh/RUBRICS.md). Preserve full outputs and disclose limitations; no metric compensates for missing required evidence.
